# ProXMax media accuracy diagnostic (Google Colab)

This standalone notebook reads one original video from Google Drive `Video/input`, its dubbed counterpart from `Video/output`, and any matching English/Bengali subtitle files it finds there. It writes **one UTF-8 `.txt` report** into `Video/output` and downloads a copy. It does not require ProXMax to be cloned or any Gemini/API key.

## Run it
1. Run each code cell from top to bottom and approve Google Drive access.
2. In the CONFIG cell, set `VIDEO_STEM` if there is more than one matching pair. The stem is the input filename without extension (for example, `lecture1`). Or set explicit video paths.
3. If subtitles have nonstandard names/locations, set `EN_SRT_PATH` and `BN_SRT_PATH`. Standard `{stem}_en.srt` / `{stem}_bn.srt` and `video_en.txt` / `video_bn.txt` names are detected automatically.
4. Downloaded copy and Drive copy are both produced; upload the `.txt` here if you want me to inspect the diagnostics.

## What the report can and cannot tell us
- Includes container/stream metadata, complete discovered subtitle text, cue counts/gaps/overlaps, English↔Bengali timestamp overlaps, and source-vs-dub audio-energy summaries for each subtitle cue.
- Audio activity uses the first audio stream, converted temporarily to mono 16 kHz, and is an **RMS energy heuristic**, not speech recognition or a reliable voice-activity detector; game audio/music can fool it. This `.txt` does not contain sound, so it cannot confirm pronunciation or audible synchronization. Keep real source pauses; use the actual media for final listening checks.
- Media processing stays in the Colab runtime and the report is saved to the mounted Drive. If FFmpeg is missing, the notebook installs it in that runtime. No API credentials are sent anywhere or written to the report; any config excerpt uses an explicit safe-field whitelist. Subtitle text may contain private dialogue, so review the report before sharing.


In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

# Standard ProXMax Drive folders.
INPUT_DIR = Path("/content/drive/MyDrive/Video/input")
OUTPUT_DIR = Path("/content/drive/MyDrive/Video/output")

# Optional: select one pair when more than one exists. Leave empty to auto-match.
VIDEO_STEM = ""  # e.g. "lecture1" for lecture1.mp4 -> lecture1_bn.mp4
SOURCE_VIDEO_PATH = ""  # optional full path override
DUBBED_VIDEO_PATH = ""  # optional full path override

# Optional subtitle overrides. Absolute paths or paths relative to OUTPUT_DIR.
EN_SRT_PATH = ""  # e.g. "lecture1_en.srt" or "video_en.txt"
BN_SRT_PATH = ""  # e.g. "lecture1_bn.srt" or "video_bn.txt"

# Optional project config.yaml path. Only a small safe whitelist of settings is reported.
CONFIG_PATH = ""

# Activity threshold; the notebook also adapts it to each audio track's noise floor.
ENERGY_THRESHOLD_DBFS = -42.0


In [ ]:
# Colab normally has NumPy and FFmpeg already. Install only if this runtime is missing them.
import shutil, subprocess, sys

if not shutil.which("ffmpeg") or not shutil.which("ffprobe"):
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", "ffmpeg"], check=True)
try:
    import numpy  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "numpy"], check=True)
print("Ready: ffmpeg, ffprobe, and NumPy are available.")


In [ ]:
import datetime as _dt
import json
import math
import re
import shutil
import subprocess
import tempfile
import wave
from pathlib import Path

import numpy as np

MEDIA_EXTS = {".mp4", ".mkv", ".mov", ".avi", ".webm", ".m4v"}
SUBTITLE_EXTS = {".srt", ".txt", ".vtt"}


def _run(command, check=True):
    result = subprocess.run(command, stdout=subprocess.PIPE, stderr=subprocess.PIPE,
                            text=True, encoding="utf-8", errors="replace")
    if check and result.returncode:
        raise RuntimeError((result.stderr or result.stdout or "Command failed")[-3000:])
    return result


def _fmt_time(seconds):
    if seconds is None or not math.isfinite(float(seconds)):
        return "n/a"
    ms = int(round(max(0.0, float(seconds)) * 1000))
    h, rem = divmod(ms, 3_600_000)
    m, rem = divmod(rem, 60_000)
    s, rem = divmod(rem, 1000)
    return f"{h:02d}:{m:02d}:{s:02d}.{rem:03d}"


def _parse_srt_time(value):
    match = re.match(r"\s*(\d{1,2}):(\d{2}):(\d{2})[,.](\d{1,3})\s*$", value)
    if not match:
        return None
    h, m, s, fraction = match.groups()
    millis = int(fraction.ljust(3, "0")[:3])
    return int(h) * 3600 + int(m) * 60 + int(s) + millis / 1000.0


def _parse_subtitles(raw):
    cues = []
    for block in re.split(r"\n\s*\n", raw.replace("\r\n", "\n").replace("\r", "\n").strip()):
        lines = block.splitlines()
        time_index = next((i for i, line in enumerate(lines) if "-->" in line), None)
        if time_index is None:
            continue
        parts = lines[time_index].split("-->", 1)
        if len(parts) != 2:
            continue
        start = _parse_srt_time(parts[0])
        end = _parse_srt_time(parts[1].strip().split()[0])
        if start is None or end is None:
            continue
        text = " ".join(line.strip() for line in lines[time_index + 1:] if line.strip())
        cues.append({"number": len(cues) + 1, "start": start, "end": end, "text": text})
    return cues


def _media_files(folder):
    folder = Path(folder)
    if not folder.is_dir():
        return []
    return sorted((p for p in folder.rglob("*") if p.is_file() and p.suffix.lower() in MEDIA_EXTS),
                  key=lambda p: str(p).lower())


def _choose_pair(input_dir, output_dir, video_stem="", source_override="", dub_override=""):
    input_dir, output_dir = Path(input_dir), Path(output_dir)
    if source_override:
        source = Path(source_override).expanduser()
        if not source.is_file():
            raise FileNotFoundError(f"SOURCE_VIDEO_PATH পাওয়া যায়নি: {source}")
    else:
        sources = _media_files(input_dir)
        if video_stem:
            sources = [p for p in sources if p.stem.casefold() == video_stem.strip().casefold()]
        if not sources:
            raise FileNotFoundError(f"Input-এ ভিডিও পাওয়া যায়নি: {input_dir}")
        outputs = _media_files(output_dir)
        pairs = []
        for src in sources:
            exact = [p for p in outputs if p.stem.casefold() == f"{src.stem}_bn".casefold()]
            if not exact:
                exact = [p for p in outputs if p.stem.casefold() == src.stem.casefold()]
            if len(exact) == 1:
                pairs.append((src, exact[0]))
        if not video_stem and len(pairs) != 1:
            if len(pairs) > 1:
                choices = "\n".join(f"  - {src.stem} -> {dub.name}" for src, dub in pairs)
                raise RuntimeError("একাধিক input/output pair আছে। উপরের CONFIG cell-এ VIDEO_STEM সেট করুন:\n" + choices)
            available = "\n".join(f"  input: {p.name}" for p in sources)
            available += "\n" + "\n".join(f"  output: {p.name}" for p in outputs)
            raise FileNotFoundError("একটি নির্দিষ্ট pair মেলেনি। CONFIG cell-এ SOURCE_VIDEO_PATH/DUBBED_VIDEO_PATH দিন।\n" + available)
        if video_stem:
            pairs = [(src, dub) for src, dub in pairs if src.stem.casefold() == video_stem.strip().casefold()]
        if len(pairs) != 1:
            raise FileNotFoundError(f"{video_stem!r}-এর জন্য একটিমাত্র output match পাওয়া যায়নি।")
        source, auto_dub = pairs[0]
    if dub_override:
        dubbed = Path(dub_override).expanduser()
        if not dubbed.is_file():
            raise FileNotFoundError(f"DUBBED_VIDEO_PATH পাওয়া যায়নি: {dubbed}")
    elif source_override:
        outputs = _media_files(output_dir)
        candidates = [p for p in outputs if p.stem.casefold() in {
            f"{source.stem}_bn".casefold(), source.stem.casefold()
        }]
        if len(candidates) != 1:
            raise FileNotFoundError("Output video মেলেনি; CONFIG cell-এ DUBBED_VIDEO_PATH দিন।")
        dubbed = candidates[0]
    else:
        dubbed = auto_dub
    return source.resolve(), dubbed.resolve()


def _resolve_optional_path(value, source, output_dir, kind):
    if value:
        path = Path(value).expanduser()
        if not path.is_absolute():
            path = Path(output_dir) / path
        return path if path.is_file() else None
    suffixes = (".srt", ".txt", ".vtt")
    names = [f"{source.stem}_{kind}{suffix}" for suffix in suffixes]
    names += [f"video_{kind}{suffix}" for suffix in suffixes]
    roots = [Path(output_dir), Path(source).parent]
    for root in roots:
        for name in names:
            candidate = root / name
            if candidate.is_file():
                return candidate
    candidates = []
    for root in roots:
        if root.is_dir():
            for candidate in root.rglob("*"):
                if candidate.is_file() and candidate.suffix.lower() in SUBTITLE_EXTS:
                    stem = candidate.stem.casefold()
                    if stem.endswith("_" + kind.casefold()):
                        candidates.append(candidate)
    candidates = sorted(set(candidates), key=lambda p: str(p).lower())
    return candidates[0] if len(candidates) == 1 else None


def _probe(path):
    cmd = ["ffprobe", "-v", "error", "-show_entries",
           "format=format_name,duration,size,bit_rate:stream=index,codec_type,codec_name,duration,width,height,avg_frame_rate,sample_rate,channels,channel_layout,bit_rate,start_time",
           "-of", "json", str(path)]
    result = _run(cmd)
    data = json.loads(result.stdout or "{}")
    fmt = data.get("format", {})
    streams = data.get("streams", [])
    return {"format": fmt, "streams": streams}


def _probe_text(label, path, info):
    fmt = info.get("format", {})
    lines = [f"{label}: {Path(path).name}",
             f"  size_bytes={Path(path).stat().st_size}",
             f"  format={fmt.get('format_name', 'n/a')}",
             f"  duration_s={fmt.get('duration', 'n/a')}",
             f"  container_bitrate={fmt.get('bit_rate', 'n/a')}"]
    for stream in info.get("streams", []):
        common = f"  stream[{stream.get('index', '?')}] {stream.get('codec_type', '?')} codec={stream.get('codec_name', 'n/a')} start_time={stream.get('start_time', 'n/a')} duration={stream.get('duration', 'n/a')} bitrate={stream.get('bit_rate', 'n/a')}"
        if stream.get("codec_type") == "video":
            common += f" size={stream.get('width', '?')}x{stream.get('height', '?')} fps={stream.get('avg_frame_rate', 'n/a')}"
        elif stream.get("codec_type") == "audio":
            common += f" sample_rate={stream.get('sample_rate', 'n/a')} channels={stream.get('channels', 'n/a')} layout={stream.get('channel_layout', 'n/a')}"
        lines.append(common)
    return lines


def _cue_stats(cues):
    if not cues:
        return {"count": 0, "gaps": [], "overlaps": [], "empty_text": 0,
                "adjacent_repeats": 0, "out_of_order": 0}
    ordered = sorted(cues, key=lambda cue: (cue["start"], cue["end"]))
    gaps, overlaps = [], []
    for left, right in zip(ordered, ordered[1:]):
        delta = right["start"] - left["end"]
        if delta > 0.001:
            gaps.append({"seconds": delta, "after": left["number"], "before": right["number"]})
        elif delta < -0.001:
            overlaps.append({"seconds": -delta, "left": left["number"], "right": right["number"]})
    lengths = [max(0.0, cue["end"] - cue["start"]) for cue in cues]
    normalized_text = [re.sub(r"\s+", " ", cue["text"]).strip().casefold() for cue in cues]
    adjacent_repeats = sum(bool(left) and left == right for left, right in zip(normalized_text, normalized_text[1:]))
    out_of_order = sum(right["start"] < left["start"] - 0.001 for left, right in zip(cues, cues[1:]))
    return {"count": len(cues), "first": ordered[0]["start"], "last": max(cue["end"] for cue in ordered),
            "total_cue_seconds": sum(lengths), "median_cue_seconds": float(np.median(lengths)),
            "max_cue_seconds": max(lengths), "gaps": gaps, "overlaps": overlaps,
            "empty_text": sum(not text for text in normalized_text),
            "adjacent_repeats": adjacent_repeats, "out_of_order": out_of_order}


def _subtitle_text(path):
    if not path:
        return "", []
    raw = Path(path).read_text(encoding="utf-8-sig", errors="replace")
    return raw, _parse_subtitles(raw)


def _extract_audio_metrics(media_path, work_dir, label, threshold_dbfs=-42.0):
    output_wav = Path(work_dir) / f"{label}.wav"
    command = ["ffmpeg", "-nostdin", "-hide_banner", "-loglevel", "error", "-y",
               "-i", str(media_path), "-map", "0:a:0", "-vn", "-ac", "1", "-ar", "16000",
               "-c:a", "pcm_s16le", str(output_wav)]
    result = _run(command, check=False)
    if result.returncode != 0 or not output_wav.is_file():
        error = (result.stderr or "Audio stream পাওয়া যায়নি")[-1200:]
        error = error.replace(str(Path(media_path).resolve()), Path(media_path).name)
        error = error.replace(str(media_path), Path(media_path).name)
        return {"error": error}
    try:
        with wave.open(str(output_wav), "rb") as wav:
            rate, channels, width, nframes = wav.getframerate(), wav.getnchannels(), wav.getsampwidth(), wav.getnframes()
            frames = wav.readframes(nframes)
        if width != 2:
            return {"error": f"Expected 16-bit PCM, got {width * 8}-bit"}
        samples = np.frombuffer(frames, dtype="<i2").astype(np.float32)
        if samples.size == 0:
            return {"error": "Audio stream empty"}
        frame_samples = max(1, int(rate * 0.1))
        levels = []
        for start in range(0, samples.size, frame_samples):
            chunk = samples[start:start + frame_samples]
            rms = float(np.sqrt(np.mean(chunk * chunk))) if chunk.size else 0.0
            levels.append(20.0 * math.log10(max(rms / 32768.0, 1e-6)))
        levels = np.asarray(levels, dtype=np.float32)
        floor = float(np.percentile(levels, 10))
        adaptive_threshold = min(-28.0, max(float(threshold_dbfs), floor + 8.0))
        quiet = levels < adaptive_threshold
        intervals = []
        run_start = None
        for i, is_quiet in enumerate(quiet.tolist() + [False]):
            if is_quiet and run_start is None:
                run_start = i
            elif not is_quiet and run_start is not None:
                start_s = run_start * 0.1
                end_s = min(samples.size / rate, i * 0.1)
                if end_s - start_s >= 0.8:
                    intervals.append((start_s, end_s, end_s - start_s))
                run_start = None
        intervals.sort(key=lambda item: item[2], reverse=True)
        return {"duration": samples.size / float(rate), "sample_rate": rate, "channels": channels,
                "rms_dbfs_p10": floor, "rms_dbfs_median": float(np.median(levels)),
                "rms_dbfs_p90": float(np.percentile(levels, 90)),
                "threshold_dbfs": adaptive_threshold, "levels": levels,
                "quiet_intervals": intervals[:80]}
    except Exception as exc:
        return {"error": f"Audio metrics failed: {type(exc).__name__}: {exc}"}


def _cue_audio_summary(cue, metrics):
    if not metrics or metrics.get("error") or "levels" not in metrics:
        return "audio=n/a"
    levels = metrics["levels"]
    left = max(0, int(cue["start"] / 0.1))
    right = min(len(levels), max(left + 1, int(math.ceil(cue["end"] / 0.1))))
    segment = levels[left:right]
    if not len(segment):
        return "audio=n/a"
    active = float(np.mean(segment >= metrics["threshold_dbfs"]))
    return f"energy_median={float(np.median(segment)):.1f}dBFS, above_threshold={active:.0%}"


def _overlapping_cue_numbers(cue, other_cues):
    found = [other["number"] for other in other_cues
             if min(cue["end"], other["end"]) - max(cue["start"], other["start"]) > 0.001]
    if found:
        return ",".join(map(str, found))
    if not other_cues:
        return "none"
    mid = (cue["start"] + cue["end"]) / 2
    nearest = min(other_cues, key=lambda other: abs((other["start"] + other["end"]) / 2 - mid))
    delta = nearest["start"] - cue["start"]
    return f"no_overlap; nearest=#{nearest['number']} start_delta={delta:+.3f}s"


def _safe_config_text(config_path):
    path = Path(config_path)
    if not path.is_file():
        return ["No config.yaml found at configured path."]
    try:
        import yaml
        data = yaml.safe_load(path.read_text(encoding="utf-8")) or {}
    except Exception as exc:
        return ["config.yaml could not be parsed; no config values were included."]
    whitelist = {
        "stt": ["model", "beam_size", "chunk_min_seconds", "chunk_max_seconds", "chunk_pad_ms", "carry_context_chars", "seam_repair"],
        "translation": ["char_rate", "shorten_pass", "expand_pass", "expand_min_gap"],
        "sync": ["min_tempo", "max_tempo", "slot_gap_ms", "borrow_silence_ms", "fill_extension"],
        "tts": ["engine", "voice", "gender", "rate", "pitch", "unit_max_seconds", "unit_max_gap"],
        "output": ["save_srt", "audio_bitrate"],
        "gemini": ["model", "batch_size", "temperature"],
    }
    lines = [f"config_path={path.name} (API keys, tokens, and paths are intentionally omitted)"]
    for section, keys in whitelist.items():
        values = data.get(section, {}) if isinstance(data, dict) else {}
        if not isinstance(values, dict):
            continue
        selected = {key: values[key] for key in keys if key in values}
        if selected:
            lines.append(f"{section}: {json.dumps(selected, ensure_ascii=False, sort_keys=True)}")
    return lines


def _append_subtitle_report(lines, label, path, raw, cues, audio_metrics):
    lines.append(f"\n[{label} subtitle stats]")
    if not path:
        lines.append("Subtitle file not found. Set EN_SRT_PATH / BN_SRT_PATH in CONFIG.")
        return
    stats = _cue_stats(cues)
    lines.append(f"file={Path(path).name}; parsed_cues={stats['count']}")
    if cues:
        lines.append(f"range={_fmt_time(stats['first'])}..{_fmt_time(stats['last'])}; total_cue_s={stats['total_cue_seconds']:.3f}; median_cue_s={stats['median_cue_seconds']:.3f}; max_cue_s={stats['max_cue_seconds']:.3f}")
        lines.append(f"gaps>0.5s={sum(g['seconds'] > .5 for g in stats['gaps'])}; >2s={sum(g['seconds'] > 2 for g in stats['gaps'])}; >5s={sum(g['seconds'] > 5 for g in stats['gaps'])}; overlaps={len(stats['overlaps'])}; repeated_adjacent_text={stats['adjacent_repeats']}; empty_text_cues={stats['empty_text']}; out_of_order_cues={stats['out_of_order']}")
        for gap in sorted(stats["gaps"], key=lambda x: x["seconds"], reverse=True)[:20]:
            lines.append(f"GAP {gap['seconds']:.3f}s after #{gap['after']} before #{gap['before']}")
        for overlap in sorted(stats["overlaps"], key=lambda x: x["seconds"], reverse=True)[:20]:
            lines.append(f"OVERLAP {overlap['seconds']:.3f}s between #{overlap['left']} and #{overlap['right']}")
        for cue in cues:
            energy = _cue_audio_summary(cue, audio_metrics)
            lines.append(f"#{cue['number']:03d} {_fmt_time(cue['start'])} --> {_fmt_time(cue['end'])} | {energy} | {cue['text']}")
    else:
        lines.append("No SRT/WebVTT cues parsed; raw text is included below for reference.")
    lines.append(f"\n[{label} raw subtitle file]")
    lines.append(raw.strip() if raw.strip() else "(empty)")


def _build_timeline_map(lines, en_cues, bn_cues, source_audio, dub_audio):
    lines.append("\n[Subtitle cue time-overlap map]")
    lines.append("Cue overlap is a timestamp comparison only; it does not prove semantic alignment.")
    if not en_cues or not bn_cues:
        lines.append("Both parsed subtitle tracks are needed for this map.")
        return
    lines.append("English cue -> Bengali cue(s) with time overlap; source audio RMS estimate:")
    for cue in en_cues:
        match = _overlapping_cue_numbers(cue, bn_cues)
        lines.append(f"EN#{cue['number']:03d} {_fmt_time(cue['start'])}-{_fmt_time(cue['end'])} -> BN {match} | {_cue_audio_summary(cue, source_audio)}")
    lines.append("Bengali cue -> English cue(s) with time overlap; dubbed audio RMS estimate:")
    for cue in bn_cues:
        match = _overlapping_cue_numbers(cue, en_cues)
        lines.append(f"BN#{cue['number']:03d} {_fmt_time(cue['start'])}-{_fmt_time(cue['end'])} -> EN {match} | {_cue_audio_summary(cue, dub_audio)}")


def build_diagnostic_report(input_dir, output_dir, video_stem="", source_override="", dub_override="",
                            en_srt_override="", bn_srt_override="", config_path="",
                            threshold_dbfs=-42.0):
    if not shutil.which("ffprobe") or not shutil.which("ffmpeg"):
        raise RuntimeError("ffmpeg/ffprobe পাওয়া যায়নি। Colab runtime-এ ffmpeg ইনস্টল করে আবার চালান।")
    source, dubbed = _choose_pair(input_dir, output_dir, video_stem, source_override, dub_override)
    en_path = _resolve_optional_path(en_srt_override, source, output_dir, "en")
    bn_path = _resolve_optional_path(bn_srt_override, source, output_dir, "bn")
    en_raw, en_cues = _subtitle_text(en_path)
    bn_raw, bn_cues = _subtitle_text(bn_path)
    source_info, dub_info = _probe(source), _probe(dubbed)

    lines = ["PROXMAX ACCURACY DIAGNOSTIC REPORT", "=" * 72,
             f"generated_local={_dt.datetime.now().astimezone().isoformat(timespec='seconds')}",
             "This report contains metadata, subtitle text/timings, and heuristic audio energy summaries.",
             "RMS analyzes the first audio stream only, converted temporarily to mono 16 kHz; temporary WAVs are not saved to Drive.",
             "It does NOT contain raw audio/video and energy is NOT a speech recognizer/VAD.",
             "No files are uploaded by this notebook; report is saved only to the mounted Drive.",
             "Review transcript text before sharing; it may contain private dialogue.",
             f"input_dir={Path(input_dir).name}; output_dir={Path(output_dir).name}",
             f"selected_stem={source.stem}"]
    lines += ["\n[Source media]"] + _probe_text("SOURCE", source, source_info)
    lines += ["\n[Dubbed media]"] + _probe_text("DUBBED", dubbed, dub_info)
    try:
        source_duration = float(source_info.get("format", {}).get("duration", 0) or 0)
        dub_duration = float(dub_info.get("format", {}).get("duration", 0) or 0)
        lines.append(f"\n[Duration comparison] source={source_duration:.3f}s dubbed={dub_duration:.3f}s delta={dub_duration - source_duration:+.3f}s")
    except Exception:
        lines.append("\n[Duration comparison] unavailable")

    with tempfile.TemporaryDirectory(prefix="proxmax_diag_") as temp_dir:
        source_audio = _extract_audio_metrics(source, temp_dir, "source", threshold_dbfs)
        dub_audio = _extract_audio_metrics(dubbed, temp_dir, "dubbed", threshold_dbfs)
        lines.append("\n[Audio energy summary]")
        for label, metrics in (("SOURCE", source_audio), ("DUBBED", dub_audio)):
            if metrics.get("error"):
                lines.append(f"{label}: audio extraction/analysis error: {metrics['error']}")
                continue
            lines.append(f"{label}: duration={metrics['duration']:.3f}s; sr={metrics['sample_rate']}; analysis_channels={metrics['channels']} (mono); dBFS p10/median/p90={metrics['rms_dbfs_p10']:.1f}/{metrics['rms_dbfs_median']:.1f}/{metrics['rms_dbfs_p90']:.1f}; low-energy threshold={metrics['threshold_dbfs']:.1f} dBFS")
            if metrics["quiet_intervals"]:
                lines.append(f"{label} low-energy intervals >=0.8s (heuristic; may miss speech or mistake effects for speech):")
                for start, end, duration in metrics["quiet_intervals"]:
                    lines.append(f"  {_fmt_time(start)}-{_fmt_time(end)} ({duration:.2f}s)")
            else:
                lines.append(f"{label}: no >=0.8s low-energy interval found at this threshold")
        _append_subtitle_report(lines, "ENGLISH", en_path, en_raw, en_cues, source_audio)
        _append_subtitle_report(lines, "BANGLA", bn_path, bn_raw, bn_cues, dub_audio)
        _build_timeline_map(lines, en_cues, bn_cues, source_audio, dub_audio)

    if config_path:
        lines.append("\n[Safe configuration subset]")
        lines.extend(_safe_config_text(config_path))
    else:
        lines.append("\n[Safe configuration subset]\nNo CONFIG_PATH supplied.")
    lines += ["\n[Interpretation cautions]",
              "- Audio RMS/low-energy estimates are not reliable VAD, especially with game audio/music.",
              "- Matching SRT time ranges does not prove that words are spoken at those times.",
              "- Keep genuine source pauses; use the original and final audio for final timing decisions."]
    report = "\n".join(lines) + "\n"
    report_path = Path(output_dir) / f"accuracy_diagnostic_{source.stem}.txt"
    report_path.parent.mkdir(parents=True, exist_ok=True)
    report_path.write_text(report, encoding="utf-8")
    return report_path, len(en_cues), len(bn_cues)


In [ ]:
# Create the report in Drive, then download a copy to your computer.
from google.colab import files

if not INPUT_DIR.is_dir():
    raise FileNotFoundError(f"Input folder not found: {INPUT_DIR}")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not CONFIG_PATH:
    candidates = [Path("/content/ProXMax/config.yaml"), Path("/content/drive/MyDrive/Video/config.yaml")]
    CONFIG_PATH = str(next((p for p in candidates if p.is_file()), ""))

report_path, english_cues, bangla_cues = build_diagnostic_report(
    input_dir=INPUT_DIR,
    output_dir=OUTPUT_DIR,
    video_stem=VIDEO_STEM,
    source_override=SOURCE_VIDEO_PATH,
    dub_override=DUBBED_VIDEO_PATH,
    en_srt_override=EN_SRT_PATH,
    bn_srt_override=BN_SRT_PATH,
    config_path=CONFIG_PATH,
    threshold_dbfs=ENERGY_THRESHOLD_DBFS,
)
print(f"Report saved to Drive: {report_path}")
print(f"Parsed cues: English={english_cues}, Bengali={bangla_cues}")
print("The report includes subtitle text; review it before sharing.")
files.download(str(report_path))
